
# End-to-End MLOps Forecasting System (Production Grade)

This notebook demonstrates a complete production-style forecasting pipeline using MLOps principles.

## Learning Objectives
By the end of this notebook, you will understand:

- End-to-end forecasting architecture
- Data ingestion and validation
- Feature engineering pipelines
- Model training and persistence
- Forecast serving and API deployment
- Monitoring and drift detection
- Automated retraining workflows



# 1. Problem Statement

Modern companies such as Walmart, Amazon, Uber Eats, and DoorDash rely on forecasting systems that continuously operate in production environments.

These systems must:

- Ingest new data automatically
- Generate predictions continuously
- Monitor model performance
- Detect drift in real time
- Trigger retraining when patterns change

The key idea is:

> The model is NOT the system. The pipeline is the system.

A production forecasting system combines:
- Data engineering
- Machine learning
- Deployment
- Monitoring
- Automation



# 2. System Architecture

The forecasting system contains the following modules:

```text
mlops-forecasting-system/

├── data_pipeline/
├── features/
├── training/
├── inference/
├── monitoring/
├── deployment/
├── pipelines/
├── models/
└── main.py
```

Each component is modular and production-oriented.



# 3. Data Pipeline

The data pipeline is responsible for:

- Data ingestion
- Data validation
- Initial preprocessing

The example below generates synthetic forecasting data.


In [ ]:

import pandas as pd
import numpy as np

class DataPipeline:

    def ingest(self):
        print("Ingesting data...")

        np.random.seed(42)
        dates = pd.date_range("2022-01-01", periods=600)

        df = pd.DataFrame({
            "date": dates,
            "promotion": np.random.choice([0,1], 600, p=[0.8,0.2]),
            "holiday": np.random.choice([0,1], 600, p=[0.95,0.05]),
        })

        df["base"] = 200 + np.sin(np.arange(600)/14)*25

        df["sales"] = (
            df["base"]
            + df["promotion"]*35
            + df["holiday"]*60
            + np.random.normal(0, 12, 600)
        )

        return df

    def validate(self, df):
        print("Validating data...")

        assert df.isnull().sum().sum() == 0
        assert df["sales"].std() > 1

        print("Data valid")
        return df



# 4. Feature Engineering

Feature engineering transforms raw data into machine learning features.

This module creates:

- Calendar features
- Lag features
- Rolling statistics


In [ ]:

class FeatureStore:

    def build_features(self, df):

        print("Creating features...")

        df = df.copy()

        df["dayofweek"] = df["date"].dt.dayofweek
        df["month"] = df["date"].dt.month

        df["lag_1"] = df["sales"].shift(1)
        df["lag_7"] = df["sales"].shift(7)

        df["rolling_mean"] = df["sales"].rolling(7).mean()

        return df.dropna()



# 5. Model Training Pipeline

The training pipeline:

- Selects features
- Trains the forecasting model
- Saves the trained artifact


In [ ]:

import os
import joblib
from sklearn.ensemble import RandomForestRegressor

class ModelPipeline:

    def train(self, df):

        features = [
            "promotion",
            "holiday",
            "dayofweek",
            "month",
            "lag_1",
            "lag_7",
            "rolling_mean"
        ]

        X = df[features]
        y = df["sales"]

        model = RandomForestRegressor(
            n_estimators=150,
            max_depth=12,
            random_state=42
        )

        model.fit(X, y)

        os.makedirs("models", exist_ok=True)

        joblib.dump(model, "models/forecast_model.pkl")

        return model, features



# 6. Forecast Inference Service

The inference layer generates predictions from trained models.


In [ ]:

class ForecastService:

    def predict(self, model, df, features):

        return model.predict(df[features])



# 7. FastAPI Deployment Layer

This API exposes the forecasting model through a REST endpoint.

Endpoint:
- `/predict`


In [ ]:

from fastapi import FastAPI
import joblib
import pandas as pd

app = FastAPI()

model = joblib.load("models/forecast_model.pkl")

features = [
    "promotion",
    "holiday",
    "dayofweek",
    "month",
    "lag_1",
    "lag_7",
    "rolling_mean"
]

@app.post("/predict")
def predict(data: dict):

    df = pd.DataFrame(data)

    preds = model.predict(df[features])

    return {"predictions": preds.tolist()}



# 8. Monitoring and Drift Detection

Monitoring ensures that the forecasting system remains reliable in production.

The system tracks:
- MAE
- RMSE
- Statistical drift
- Distribution changes


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import ks_2samp

class Metrics:

    def compute(self, y_true, y_pred):

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        print(f"MAE: {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")

        return {"mae": mae, "rmse": rmse}


class DriftDetector:

    def compute_psi(self, train, test):

        train_mean = np.mean(train)
        test_mean = np.mean(test)

        psi = abs(train_mean - test_mean) / (train_mean + 1e-6)

        return psi

    def compute_ks(self, train, test):

        stat, _ = ks_2samp(train, test)

        return stat

    def detect(self, train_df, test_df):

        train = train_df["sales"]
        test = test_df["sales"]

        drift = abs(train.mean() - test.mean())

        psi = self.compute_psi(train, test)
        ks_stat = self.compute_ks(train, test)

        embedding_distance = abs(train.std() - test.std())

        print(f"Mean Drift: {drift:.2f}")
        print(f"PSI: {psi:.4f}")
        print(f"KS Statistic: {ks_stat:.4f}")
        print(f"Embedding Distance: {embedding_distance:.4f}")

        if (
            drift > 12 or
            psi > 0.2 or
            ks_stat > 0.1 or
            embedding_distance > 5
        ):
            print("DATA DRIFT DETECTED")
            return True

        return False



# 9. Automated Retraining

When drift is detected, the system can automatically trigger retraining workflows.


In [ ]:

class RetrainingPipeline:

    def trigger(self, drift_flag):

        if drift_flag:
            print("Triggering retraining pipeline...")
        else:
            print("System stable - no retraining needed")



# 10. Main Orchestrator

The orchestrator connects all components into one production workflow.


In [ ]:

# INIT
data_pipeline = DataPipeline()
feature_store = FeatureStore()
model_pipeline = ModelPipeline()

metrics = Metrics()
drift_detector = DriftDetector()
retrain = RetrainingPipeline()

# DATA
df = data_pipeline.ingest()
df = data_pipeline.validate(df)

# FEATURES
df = feature_store.build_features(df)

# SPLIT
split = int(len(df) * 0.8)

train = df[:split]
test = df[split:]

# TRAIN
model, features = model_pipeline.train(train)

# PREDICT
service = ForecastService()

preds = service.predict(model, test, features)

# EVALUATE
metrics.compute(test["sales"], preds)

# DRIFT
drift_flag = drift_detector.detect(train, test)

# RETRAIN
retrain.trigger(drift_flag)



# 11. Docker Deployment

The following Docker configuration can containerize the forecasting API for production deployment.


In [ ]:

FROM python:3.10

WORKDIR /app

COPY . /app

RUN pip install fastapi uvicorn scikit-learn pandas joblib

CMD ["uvicorn", "inference.api:app", "--host", "0.0.0.0", "--port", "8000"]



# 12. Summary

This notebook demonstrated a complete production-style forecasting system using MLOps principles.

Key concepts covered:
- Modular architecture
- Feature engineering
- Forecast training
- API deployment
- Monitoring and drift detection
- Automated retraining
- Docker deployment

This structure represents how modern real-world forecasting systems are designed in production environments.
